### 检查 3 个文件的表头是否一致

In [1]:
import pandas as pd
from pathlib import Path

base = Path(r"hblim-customer-complaints")

files = [
    base / "train-00000-of-00001.parquet",
    base / "validation-00000-of-00001.parquet",
    base / "test-00000-of-00001.parquet",
]

# 读取 3 个 parquet
dfs = {}
for f in files:
    name = f.name
    dfs[name] = pd.read_parquet(f)
    print(f"{name} shape: {dfs[name].shape}")

# 获取每个文件的表头
headers = {name: list(df.columns) for name, df in dfs.items()}

# 逐个比较
ref_name, ref_cols = next(iter(headers.items()))
all_same = True
for name, cols in headers.items():
    if cols != ref_cols:
        all_same = False
        print(f"\n表头不一致: {name}")
        print(f"参考文件: {ref_name}")
        print(f"{name} 多出的列: {[c for c in cols if c not in ref_cols]}")
        print(f"{name} 缺少的列: {[c for c in ref_cols if c not in cols]}")
        print(f"{name} 列表: {cols}")
        print(f"{ref_name} 列表: {ref_cols}")
        break

if all_same:
    print("\n✅ 3 个文件的表头一致")
    print(ref_cols)

train-00000-of-00001.parquet shape: (1261, 2)
validation-00000-of-00001.parquet shape: (210, 2)
test-00000-of-00001.parquet shape: (211, 2)

✅ 3 个文件的表头一致
['text', 'label']


In [ ]:
dfs = [pd.read_parquet(f) for f in files]

all_cols = sorted({c for df in dfs for c in df.columns})
merged = pd.concat(
    [df.reindex(columns=all_cols) for df in dfs],
    ignore_index=True,
    sort=False
)

# 先保证 label 是数字
merged["label"] = pd.to_numeric(merged["label"], errors="coerce")

# 按 label 映射到 complaint_type
label_to_complaint_type = {
    0: "billing",
    1: "delivery",
    2: "product"
}

merged["complaint_type"] = merged["label"].map(label_to_complaint_type)

out = base / "customer-complaints-merged.parquet"
merged.to_parquet(out, index=False)
print(f"合并完成，共 {len(merged)} 行")

合并完成，共 1682 行
